# SIH26142 — Real-ESRGAN Fine-Tuning on Sentinel-2 Pairs
## Colab/Kaggle Backup Notebook

**Use this notebook if you cannot run `src/train.py` locally.**
If you have a local GPU, use `python src/train.py` instead — it's equivalent.

### Steps:
1. Run all cells in order
2. When training completes, download `model_finetuned_best.pth`
3. Place it in `srm-project/src/checkpoints/`
4. Select it in the dashboard sidebar


In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Training on CPU will be impractically slow.')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────
!pip install -q basicsr facexlib gfpgan realesrgan rasterio scikit-image tqdm

In [ ]:
# ── Cell 3: Mount Google Drive (to persist checkpoints) ──────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/srm_checkpoints'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_DIR}')

In [ ]:
# ── Cell 4: Clone repo or upload training pairs ──────────────────────────
# OPTION A: Clone your GitHub repo
# !git clone https://github.com/<your-username>/srm-project.git
# %cd srm-project

# OPTION B: Upload pairs ZIP manually via Files panel, then unzip:
# from google.colab import files
# uploaded = files.upload()  # Upload data/synthetic_pairs.zip
# !unzip synthetic_pairs.zip -d data/

# For this demo, we'll generate synthetic pairs on the fly from a sample tile
import numpy as np, os
from pathlib import Path

PAIRS_DIR = Path('/content/synthetic_pairs')
(PAIRS_DIR / 'lr').mkdir(parents=True, exist_ok=True)
(PAIRS_DIR / 'hr').mkdir(parents=True, exist_ok=True)

# Generate synthetic pairs from random data (replace with real pairs from repo)
rng = np.random.default_rng(42)
N_PAIRS = 200
for i in range(N_PAIRS):
    hr = rng.random((3, 512, 512), dtype=np.float32)
    import cv2
    lr_ch = [cv2.resize(hr[c], (128, 128), interpolation=cv2.INTER_CUBIC) for c in range(3)]
    lr = np.stack(lr_ch, axis=0)
    np.save(PAIRS_DIR / 'hr' / f'patch_{i:04d}_hr.npy', hr)
    np.save(PAIRS_DIR / 'lr' / f'patch_{i:04d}_lr.npy', lr)

print(f'Generated {N_PAIRS} synthetic training pairs in {PAIRS_DIR}')

In [ ]:
# ── Cell 5: Download pretrained weights ──────────────────────────────────
import urllib.request
WEIGHTS_URL = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
WEIGHTS_PATH = '/content/RealESRGAN_x4plus.pth'
if not os.path.exists(WEIGHTS_PATH):
    print('Downloading pretrained weights ...')
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
    print('Done.')
else:
    print('Weights already present.')

In [ ]:
# ── Cell 6: Load generator ────────────────────────────────────────────────
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

generator = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=4
)
state_dict = torch.load(WEIGHTS_PATH, map_location='cpu')
if 'params_ema' in state_dict:
    state_dict = state_dict['params_ema']
elif 'params' in state_dict:
    state_dict = state_dict['params']
generator.load_state_dict(state_dict, strict=True)
generator = generator.to(device)
print(f'Generator loaded on {device}')

In [ ]:
# ── Cell 7: Training configuration ───────────────────────────────────────
EPOCHS = 50          # Increase for better results (100-200 recommended)
BATCH_SIZE = 4       # Reduce if OOM
CROP_SIZE = 128      # LR crop size (HR crop = 512)
LEARNING_RATE = 1e-4
LAMBDA_PERCEPTUAL = 0.1
SAVE_EVERY = 10

In [ ]:
# ── Cell 8: Dataset ───────────────────────────────────────────────────────
import random
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class ColabPairDataset(Dataset):
    def __init__(self, pairs_dir, split='train', val_frac=0.1, crop_size=128):
        self.lr_files = sorted((Path(pairs_dir) / 'lr').glob('*_lr.npy'))
        rng = random.Random(42)
        rng.shuffle(self.lr_files)
        n_val = max(1, int(len(self.lr_files) * val_frac))
        self.lr_files = self.lr_files[:n_val] if split == 'val' else self.lr_files[n_val:]
        self.crop_size = crop_size

    def __len__(self): return len(self.lr_files)

    def __getitem__(self, idx):
        lrf = self.lr_files[idx]
        hrf = Path(str(lrf).replace('/lr/', '/hr/').replace('_lr.npy', '_hr.npy'))
        lr = torch.from_numpy(np.load(lrf)).float()[:3]
        hr = torch.from_numpy(np.load(hrf)).float()[:3]
        c, lh, lw = lr.shape
        cs = self.crop_size
        if lh >= cs and lw >= cs:
            t = torch.randint(0, lh - cs + 1, (1,)).item()
            l = torch.randint(0, lw - cs + 1, (1,)).item()
        else:
            t, l = 0, 0
            cs = min(lh, lw)
        lr_c = lr[:, t:t+cs, l:l+cs]
        hr_c = hr[:, t*4:(t+cs)*4, l*4:(l+cs)*4]
        if torch.rand(1) > 0.5:
            lr_c = torch.flip(lr_c, [2])
            hr_c = torch.flip(hr_c, [2])
        return lr_c, hr_c

train_ds = ColabPairDataset(PAIRS_DIR, 'train', crop_size=CROP_SIZE)
val_ds   = ColabPairDataset(PAIRS_DIR, 'val',   crop_size=CROP_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# ── Cell 9: Perceptual loss ───────────────────────────────────────────────
import torchvision.models as models

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.feat = nn.Sequential(*list(vgg.features)[:16]).to(device).eval()
        for p in self.feat.parameters(): p.requires_grad = False
    def forward(self, pred, tgt):
        mean = torch.tensor([0.485,0.456,0.406], device=device).view(1,3,1,1)
        std  = torch.tensor([0.229,0.224,0.225], device=device).view(1,3,1,1)
        norm = lambda x: (x - mean) / std
        return F.l1_loss(self.feat(norm(pred)), self.feat(norm(tgt.detach())))

l1_loss    = nn.L1Loss()
perc_loss  = PerceptualLoss()
optimizer  = torch.optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.99))
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
print('Optimizer and losses ready')

In [ ]:
# ── Cell 10: Training loop ────────────────────────────────────────────────
import time, json
from IPython.display import clear_output
import matplotlib.pyplot as plt

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
CKPT_BEST  = f'{DRIVE_DIR}/model_finetuned_best.pth'
CKPT_FINAL = f'{DRIVE_DIR}/model_finetuned_final.pth'

for epoch in range(1, EPOCHS + 1):
    generator.train()
    t0 = time.time()
    train_losses = []

    for lr_b, hr_b in train_loader:
        lr_b, hr_b = lr_b.to(device), hr_b.to(device)
        optimizer.zero_grad()
        sr_b   = generator(lr_b)
        loss   = l1_loss(sr_b, hr_b) + LAMBDA_PERCEPTUAL * perc_loss(sr_b, hr_b)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    scheduler.step()

    generator.eval()
    val_losses = []
    with torch.no_grad():
        for lr_b, hr_b in val_loader:
            sr_b = generator(lr_b.to(device))
            val_losses.append(l1_loss(sr_b, hr_b.to(device)).item())

    mt = np.mean(train_losses)
    mv = np.mean(val_losses) if val_losses else float('nan')
    history['train_loss'].append(mt)
    history['val_loss'].append(mv)

    if mv < best_val_loss:
        best_val_loss = mv
        torch.save({'params_ema': generator.state_dict(), 'epoch': epoch, 'val_loss': mv}, CKPT_BEST)

    if epoch % SAVE_EVERY == 0:
        clear_output(wait=True)
        plt.figure(figsize=(8,3))
        plt.plot(history['train_loss'], label='Train')
        plt.plot(history['val_loss'],   label='Val', linestyle='--')
        plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Training Progress')
        plt.tight_layout(); plt.show()

    print(f'Epoch {epoch:3d}/{EPOCHS} | Train: {mt:.4f} | Val: {mv:.4f} | {time.time()-t0:.1f}s')

# Save final
torch.save({'params_ema': generator.state_dict(), 'epoch': EPOCHS, 'history': history}, CKPT_FINAL)
with open(f'{DRIVE_DIR}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'\nTraining complete. Best: {CKPT_BEST}')

In [ ]:
# ── Cell 11: Download checkpoint ─────────────────────────────────────────
from google.colab import files
files.download(CKPT_BEST)
files.download(f'{DRIVE_DIR}/training_history.json')
print('Downloaded. Place these in srm-project/src/checkpoints/')